In [30]:
import numpy as np
import pandas as pd
import torch

In [49]:
gaze_dir  = "data/Varjogaze/"
trial_dir = "data/trial_data/"

def load_participant_data(num):
    gaze_df   = pd.read_csv(gaze_dir + f"user_{num}VarjoGaze.csv",na_values=["nan(ind)", "-nan(ind)", "inf", "-inf"], low_memory=False)
    gaze_df = gaze_df.replace([np.inf, -np.inf],np.nan)
    trial_df = pd.read_csv(trial_dir + f"user_{num}.csv", low_memory=False)

    return gaze_df, trial_df

In [50]:
def process_participant(gaze_df, trial_df):

    # convert nanosec measurements to milisec to align with trial data
    gaze_df["timestamp_ms"] = (
        gaze_df["relative_to_unix_epoch_timestamp"] / 1e6
    )

    gaze_df["trial_type"] = -1 # placeholder

    for _, trial in trial_df.iterrows():

        mask = (
            (gaze_df["timestamp_ms"] >= trial["StartTime"])
            &
            (gaze_df["timestamp_ms"] <= trial["EndTime"])
        )

        gaze_df.loc[mask, "trial_type"] = trial["TrialType"]

    # Simplifying labels into just high and low cognitive load
    LABEL_MAP = {
        0: 0,  # low
        # 2: 0,  # low
        # 1: 1,  # high
        3: 1   # high
    }

    relaxation_df = gaze_df[gaze_df["trial_type"] == 5].copy()

    training_df = gaze_df[
        gaze_df["trial_type"].isin([0, 3])#([0,1,2,3])
    ].copy()

    training_df["label"] = (
        training_df["trial_type"]
        .map(LABEL_MAP)
    )

    # Remove invalid values (0.0)
    relaxation_df = relaxation_df[relaxation_df["status"] == 2]
    
    relaxation_df = relaxation_df[
        relaxation_df["left_pupil_diameter_in_mm"] > 0
    ].copy()

    relaxation_df = relaxation_df[
        relaxation_df["right_pupil_diameter_in_mm"] > 0
    ].copy()

    # Normalization
    baseline_left = (
        relaxation_df["left_pupil_diameter_in_mm"]
        .mean()
    )

    baseline_right = (
        relaxation_df["right_pupil_diameter_in_mm"]
        .mean()
    )

    training_df["left_pupil_norm"] = (
        training_df["left_pupil_diameter_in_mm"]
        / baseline_left
    )

    training_df["right_pupil_norm"] = (
        training_df["right_pupil_diameter_in_mm"]
        / baseline_right
    )

    training_df = training_df.sort_values("timestamp_ms").reset_index(drop=True)

    training_df = training_df[training_df["status"] == 2]

    dt = (
        training_df["timestamp_ms"]
        .diff()
        .median()
    )

    WINDOW_SECONDS = 2.0
    SAMPLE_RATE = 1000 / dt

    samples_per_window = int(
        WINDOW_SECONDS * SAMPLE_RATE
    )

    STEP = samples_per_window // 2

    training_df["dt"] = training_df["timestamp_ms"].diff()
    training_df["segment_id"] = (training_df["dt"] > training_df["dt"].median() * 10).cumsum()


    return training_df#, samples_per_window, STEP

In [51]:
def generate_windows(
    df,
    features,
    window_size = 400,
    step_size = 200
):

    X = []
    y = []

    for _, segment in df.groupby("segment_id"):

        values = segment[features].values
        labels = segment["label"].values

        if len(segment) < window_size:
            continue

        for start in range(0, len(segment) - window_size + 1, step_size):

            end = start + window_size

            window_labels = labels[start:end]

            label = int(window_labels.mean() > 0.5)

            X.append(values[start:end])
            y.append(label)

    # values = df[features].values
    # labels = df["label"].values

    # for start in range(
    #     0,
    #     len(df) - window_size,
    #     step_size
    # ):

    #     end = start + window_size

    #     window = values[start:end]

    #     window_labels = labels[start:end]

    #     label = (
    #         int(window_labels.mean() > 0.5)
    #     )

    #     X.append(window)
    #     y.append(label)

    return (
        np.array(X),
        np.array(y)
    )

In [55]:
FEATURES = [
    "left_pupil_norm",
    "right_pupil_norm",

    "gaze_forward_x",
    "gaze_forward_y",
    "gaze_forward_z",

    # "focus_distance",

    "left_eye_openness",
    "right_eye_openness"
]

def preprocess_participant_data(participant_id, features):
    gaze_df, trial_df = load_participant_data(participant_id)
    training_df = process_participant(gaze_df, trial_df) #, samples_per_window, STEP 
    # print(training_df[FEATURES].isna().sum()) # Used to find errors in pupil measurements
    X, y = generate_windows(training_df, features)#, samples_per_window, STEP)
    participant_ids = np.full(len(X), participant_id)

    return X, y, participant_ids

Check valid rows

In [53]:
gaze_df_test, trial_df_test = load_participant_data(12)

print(12, gaze_df_test["status"].value_counts()[0] / gaze_df_test["status"].value_counts()[2] * 100,
          gaze_df_test["left_status"].value_counts()[0] / gaze_df_test["left_status"].value_counts()[2] * 100,
          gaze_df_test["right_status"].value_counts()[0] / gaze_df_test["right_status"].value_counts()[2] * 100)

12 31.685269147282842 32.1465949691436 32.431308319469274


In [54]:
for num in range(12, 48):
    try:
        gaze_df_test, trial_df_test = load_participant_data(num)

        print(num, gaze_df_test["status"].value_counts()[0] / gaze_df_test["status"].value_counts()[2] * 100,
                   gaze_df_test["left_status"].value_counts()[0] / gaze_df_test["left_status"].value_counts()[2] * 100,
                   gaze_df_test["right_status"].value_counts()[0] / gaze_df_test["right_status"].value_counts()[2] * 100)
    except FileNotFoundError:
        continue

12 31.685269147282842 32.1465949691436 32.431308319469274
13 5.512720280898721 11.467946136200466 6.878594115095633


KeyboardInterrupt: 

Exclude participant 12 for high amount of invalid tracking data

Remove invalid rows

In [7]:
gaze_df_test, trial_df_test = load_participant_data(13)
training_df_test, samples_per_window, STEP = process_participant(gaze_df_test, trial_df_test)

In [8]:
print(training_df_test["dt"].describe())

count    189854.000000
mean          7.561027
std         630.421984
min           4.174072
25%           5.004395
50%           5.012695
75%           5.021484
max      162605.805420
Name: dt, dtype: float64


In [9]:
training_df_test["dt"].quantile(0.99)

np.float64(5.111083984375)

In [14]:
training_df_test.head()

,raw_timestamp,relative_to_unix_epoch_timestamp,relative_to_video_first_frame_timestamp,focus_distance,frame_number,stability,status,gaze_forward_x,gaze_forward_y,gaze_forward_z,...,right_pupil_diameter_in_mm,right_pupil_iris_diameter_ratio,right_eye_openness,timestamp_ms,trial_type,label,left_pupil_norm,right_pupil_norm,dt,segment_id
0,1000109617740087900,1734449593481991500,471307444900,2.0,408592,0.759830,2,0.171653,0.059632,0.983351,...,0.0,0.0,0.987979,1.734450e+12,3,1,0.0,0.0,NaN,0
1,1000109617745080400,1734449593486984000,471312437400,2.0,408593,0.781924,2,0.171340,0.059741,0.983399,...,0.0,0.0,0.987939,1.734450e+12,3,1,0.0,0.0,4.992432,0
2,1000109617750102800,1734449593492006400,471317459800,2.0,408594,0.785341,2,0.171045,0.059640,0.983456,...,0.0,0.0,0.988055,1.734450e+12,3,1,0.0,0.0,5.022461,0
3,1000109617755239800,1734449593497143400,471322596800,2.0,408595,0.781841,2,0.171256,0.059307,0.983440,...,0.0,0.0,0.988310,1.734450e+12,3,1,0.0,0.0,5.136963,0
4,1000109617760256900,1734449593502160500,471327613900,2.0,408596,0.774421,2,0.170332,0.058611,0.983642,...,0.0,0.0,0.988346,1.734450e+12,3,1,0.0,0.0,5.017090,0


Create dataset

In [56]:
all_X = []
all_y = []
all_participants = []

participants_num = [x for x in range(13, 48) if x != 42] # Excluding participant 12 due to high amount of invalid rows. Also excluding participant 42 due to errors in pupil measurements

FEATURES = [
    "left_pupil_norm",
    "right_pupil_norm",

    "gaze_forward_x",
    "gaze_forward_y",
    "gaze_forward_z",

    # "focus_distance",

    "left_eye_openness",
    "right_eye_openness"
]

for participant_id in participants_num:
    try:
        X, y, participant_ids = preprocess_participant_data(participant_id, FEATURES)
    except FileNotFoundError:
        continue # participant is not in list

    all_X.append(X)
    all_y.append(y)
    all_participants.append(participant_ids)

    print(f"Participant {participant_id} done\n")


X = np.concatenate(all_X)
y = np.concatenate(all_y)
participants = np.concatenate(all_participants)

Participant 13 done

Participant 14 done

Participant 15 done

Participant 16 done

Participant 18 done

Participant 19 done

Participant 20 done

Participant 21 done

Participant 22 done

Participant 23 done

Participant 24 done

Participant 25 done

Participant 26 done

Participant 27 done

Participant 28 done

Participant 29 done

Participant 30 done

Participant 31 done

Participant 32 done

Participant 33 done

Participant 34 done

Participant 35 done

Participant 36 done

Participant 37 done

Participant 38 done

Participant 39 done

Participant 40 done

Participant 41 done

Participant 43 done

Participant 44 done

Participant 45 done

Participant 46 done

Participant 47 done



In [47]:
for i in range(len(X)):
    if np.count_nonzero(np.isnan(X[i])) > 0:
        print(i, np.count_nonzero(np.isnan(X[i])))
    else:
        continue

In [57]:
print(len(X))

13609


In [24]:
all_X[0].shape

(872, 400, 7)

In [25]:
X.shape

(27852, 400, 7)

In [26]:
all_y[0].shape

(872,)

In [27]:
participants.shape

(27852,)

In [58]:
# Save dataset to be used with a PyTorch model
torch.save(
    {
        "X": X,
        "y": y,
        "participant": participants
    },
    "cognitive_load_dataset_2.pt"
)